# FLAG 2027 - Notebook 20: FINAL PIPELINE (raw files -> features -> frozen heads -> checked ZIPs)

Two modes, set in Cell 1:
- `MODE = 'dev'`   : rehearsal. Scores the dev lists from the RAW jpg/wav files and checks that the result matches our cached-feature scores.
- `MODE = 'eval'`  : evaluation phase. Put ONE zip in the Drive folder `FLAG2027/eval/` and run. Only system **E** (the entry) is scored; one zip comes out.

**v9 (5 Nov, eval-phase hardening).** Everything an eval run writes is tagged with the zip's own name: features go to `final/feats_eval_<zipname>_<split>.npz`, the submission to `submissions/20_eval_<zipname>_s21_E.zip`. So a dry run on a renamed copy of `dev_set.zip` can never be cached as, or mistaken for, the real evaluation set. Cell 5 refuses to run if `FLAG2027/eval/` holds more than one zip, finds the four pair lists even if their names carry neither 'test' nor 'eval', picks the voice and face columns by file extension, and prints the first line of every list. In eval mode Cell 7 compares the scores with the dev run of E whenever the pair counts match the dev set (the dry-run check: expected correlation 1.0000) and prints the line count of every file inside the zip.

**Dry run (do once, before 11 Nov):** copy `dev_set.zip` into `FLAG2027/eval/` under the name `dryrun_dev_set.zip`, set `MODE = 'eval'`, Run all (about 45 min, T4), paste the SUMMARY. Expected: 4 lists found, ZIP VALID, dry-run check 1.0000 in all four splits. Then delete `dryrun_dev_set.zip` from `FLAG2027/eval/`. The zip and features it produced carry `dryrun_dev_set` in their names and can stay.

**Evaluation (11 Nov):** put the organizers' zip in `FLAG2027/eval/` (the only zip there), `MODE = 'eval'`, Run all, paste the SUMMARY, upload `submissions/20_eval_<zipname>_s21_E.zip` once, screenshot the accepted status.

**v10 (5 Nov, research round 9).** Three new systems on the same frozen features, all judged on the dev board: **E_novox** (E trained without any VoxCeleb), **E_sel500** and **E_sel900** (E trained with only the 500 / 900 of the ~3,700 VoxCeleb identities, four parts, whose face and voice centroids are nearest to the MAV + challenge training population; a training-side data-curation step, declared). Reason: VoxCeleb carries nationality and accent directions that are useless inside a single-population test set, and our gender-Bangla cell worsened monotonically with VoxCeleb size. Cell 1 has `EVAL_ENTRY` (the system scored in eval mode). Dev run: about 45 min (three heads, 21 seeds each), upload `20_dev_s21_E_novox.zip`, `20_dev_s21_E_sel500.zip`, `20_dev_s21_E_sel900.zip`.

**v11 (7 Nov, NB41).** One new system, **J** = E's recipe and E's data (VoxCeleb test + part 1) with the **LoRA-tuned CLIP** from NB41 replacing frozen CLIP as the face block. NB41's CV: better than E's recipe on both fold sets and in both cells (stage A âˆ’0.50, stage B âˆ’0.14), short of the âˆ’0.4 adoption bar, so J is judged on the dev board against E (22.16, same identities). Cell 4 carries the LoRA encoder (`final/clip_ft.pt`) so dev and eval extraction stay inside this notebook; Cell 3 trains the J head (21 seeds) if `final/head_J_s21.pt` is missing. In dev mode v11 scores E, E2 and J only (the other systems are all rejected and their zips exist). Upload `20_dev_s21_J.zip` once. If J becomes the entry, set `EVAL_ENTRY = 'J'` in Cell 1 and repeat the dry run (the LoRA encoder then runs on the evaluation faces).

Systems (fixed max-separation prototypes; score = -L2 of tanh embeddings; fusion = raw average, no normalisation):
- **A** : face CLIP | voice ECAPA + WavLM-sv + ResNet293 + ReDimNet2 + audeering; challenge + MAV v1/v2/v3 + VoxCeleb2; batch 64x8, cosine LR (dev 28.09)
- **C** [v4 29-Oct] : A + Whisper-large-v3 encoder block + NAP-8 language-nuisance projection (dev 26.70)
- **D** [v5 3-Nov] : C + symmetric InfoNCE alignment between the two scored embeddings (dev 22.70)
- **D16** [v5 3-Nov] : the same loss with NB28's 16-id recipe (dev 23.32)
- **E** [v6 4-Nov] : D with the SupCon 'L_out' loss form (dev 22.16, **the evaluation entry**, `final/head_E_s21.pt`)
- **E2** [v6 4-Nov] : E + VoxCeleb2 part 2 identities, 2,626 training identities (dev 21.92, within noise of E)
- **G** / **G16** [v7 5-Nov] : E2 data + NB34 S28 head settings (dev 22.35 / 23.07; rejected)
- **E3** [v8 5-Nov] : E + VoxCeleb2 parts 2 AND 3 (dev 22.69; rejected, gender-Bangla worsens with VoxCeleb size)
- **E_nap24** [v8 5-Nov] : E with 24 language directions removed (dev 23.02; rejected)
- **E+E2** [v6b] : raw average of the E and E2 scores (dev 21.97; noise)
- **J** [v11 7-Nov] : E with NB41's LoRA-tuned CLIP as the face block (same identities as E); judged on the dev board against E

Heads are trained ONCE and saved to `FLAG2027/final/` (skipped if present). Nothing trains in v9: all heads exist. In dev mode all systems are still scored, as before.


In [ ]:
# Cell 1 [v12 19-Nov, as used for the evaluation submissions] - MODE, EVAL_ENTRY, Drive, GPU (hard check), packages (v11: + peft for the LoRA-tuned CLIP of system J; Colab's torchao removed first, as in NB41)
MODE = 'eval'            # 'dev' = rehearsal on the dev lists | 'eval' = the ONE zip in Drive FLAG2027/eval/ (the evaluation set, or dryrun_dev_set.zip for the dry run)
EVAL_ENTRY = 'E2'       # the system scored in eval mode (E: dev 22.16). Change only when a better system is confirmed on the dev board (v11: 'J' if it beats E there).
from google.colab import drive
drive.mount('/content/drive')
!pip -q uninstall -y torchao
!pip -q install speechbrain onnx onnx2torch peft
import torch, shutil
assert MODE in ('dev', 'eval'), "MODE must be 'dev' or 'eval'"
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run again.'
dev = 'cuda'
print('[v11 7-Nov] MODE:', MODE, '| eval entry:', EVAL_ENTRY, '| Device:', torch.cuda.get_device_name(0), '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))


In [ ]:
# Cell 2 [v11 7-Nov] - paths, model code, helpers, SYSTEMS (A, C, D, E, E2, G, E3, E_nap24, E_novox, E_sel500, E_sel900; v11: J = E with the LoRA-tuned CLIP face block)
# v11: cfg 'face' names the face block ('clip' = frozen CLIP ViT-L/14, the default for every older system; 'clip_ft' = NB41's LoRA-tuned CLIP)
import os, re, json, time, zipfile, copy, glob
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
import librosa, soundfile as sf
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
FINAL = os.path.join(PROJECT, 'final'); os.makedirs(FINAL, exist_ok=True)
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128, p=0.5):                # [v7] width d and dropout p configurable (old systems: 128 / 0.5)
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(p))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128, p=0.5):
        super().__init__()
        self.face, self.voice = Branch(d_face, d, p), Branch(d_voice, d, p)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float(); pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """[v5] symmetric identity-aware InfoNCE between the two SCORED embeddings (face tanh vs voice tanh); every same-identity row is a positive.
    'L_in' form. Identical to NB28 H2 (CV 23.96 vs 25.18: std -0.78, gender -1.64)."""
    S_ = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(M):
        logp = M - torch.logsumexp(M, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S_) + one(S_.t()))
def lout_loss(ft, vt, lab, tau):
    """[v6] SupCon 'L_out' form (Khosla 2020): mean over positives of the log-probability, symmetric face->voice and voice->face.
    Identical to NB33 c_lout (CV stage B 22.85 vs 23.41 for L_in: std -0.49, gender -0.61)."""
    S_ = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P_ = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P_ * Fn.log_softmax(M, 1)).sum(1).div(P_.sum(1)).mean()
    return 0.5 * (one(S_) + one(S_.t()))
ALIGN_FN = {'nce': nce_loss, 'lout': lout_loss}
# [v8] NB37 training hooks, copied unchanged from NB37 cell 3; each is used only when the system's cfg asks for it (sf2 / hinge / pair / mix)
def sf2_loss(z, P, lab, bias, s=16.0, m=0.1, t=3.0, lam=0.7):
    """SphereFace2: one binary decision per prototype; positive prototype pulled past +m, every other prototype pushed below -m"""
    cos = Fn.normalize(z, dim=1) @ P.t(); g = 2 * ((cos + 1) / 2) ** t - 1
    onehot = Fn.one_hot(lab, g.shape[1]).float()
    pos = Fn.softplus(-s * (g - m) - bias) * onehot; neg = Fn.softplus(s * (g + m) + bias) * (1 - onehot)
    return (lam * pos.sum(1) + (1 - lam) * neg.sum(1) / (g.shape[1] - 1)).mean()
def hinge_loss(ft, vt, m=1 - np.cos(np.deg2rad(30))):
    return Fn.relu(1 - (Fn.normalize(ft, dim=1) * Fn.normalize(vt, dim=1)).sum(1) - m).mean()
class PairHead(nn.Module):
    def __init__(self, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(4 * d, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, f, v):
        return self.net(torch.cat([f, v, f * v, (f - v).abs()], 1)).squeeze(1)
def pair_bce(ph, ft, vt, lab):
    n = len(lab); same = lab[:, None] == lab[None, :]
    pi, pj = torch.where(same); ni, nj = torch.where(~same)
    sel = torch.randperm(len(ni), device=lab.device)[:len(pi)]; ni, nj = ni[sel], nj[sel]
    logits = torch.cat([ph(ft[pi], vt[pj]), ph(ft[ni], vt[nj])]); y = torch.cat([torch.ones(len(pi)), torch.zeros(len(ni))]).to(lab.device)
    return Fn.binary_cross_entropy_with_logits(logits, y)
def make_model(cfg, n_class, d_face, d_voice):
    """[v8] FOP with the optional NB37 parts attached, so that saved state dicts load back into an identical module"""
    m = FOP(n_class, d_face, d_voice, cfg.get('d', 128), cfg.get('p_br', 0.5))
    if cfg.get('pair'):
        m.pair = PairHead(cfg.get('d', 128))
    if cfg.get('sf2'):
        m.sf2_bias = nn.Parameter(torch.zeros(1))
    return m
@torch.no_grad()
def head_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()          # HIGHER = same person
@torch.no_grad()
def pair_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return model.pair(ft, vt).cpu().numpy()               # HIGHER = same person (pair-classifier logit)
def score_fn(cfg):
    return pair_score if cfg.get('pair') else head_score
# NAP (nuisance-attribute projection, NB24): per bilingual MAV TRAINING speaker, mean(language A) - mean(language B); top-k right singular
# vectors are 'language' directions, projected out of the (l2-per-block, concatenated) voice vector BEFORE domain normalisation. Fixed before
# test time; no test data involved. Identical to NB24's nap_basis / nap_apply. [v8] k per system via cfg['nap_k'] (default 8).
NAP_K = 8
def nap_basis(V_ext, spk, lang, k=NAP_K):
    diffs = []
    for s in np.unique(spk):
        m = spk == s; langs = np.unique(lang[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (lang == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']
# [v7] NB34 S28 head settings (CV stage B 21.76 vs 22.85 for E's settings: std -1.03, gender -1.15) -> dev G 22.35 / G16 23.07, rejected
S28 = dict(align='lout', w=0.25, tau=0.05, d=512, p_br=0.7, alpha=1.0, lr=1e-4, p_in=0.2)
E_BASE = dict(voice=R4 + ['whisper'], p_id=64, cosine=True, nap=True, csv='nb18_cv.csv', variant='V1_batch64_cosine', col='variant', align='lout', w=0.5, tau=0.07)
SYSTEMS = {'A':   dict(voice=R4,               p_id=64, cosine=True,  nap=False, csv='nb18_cv.csv', variant='V1_batch64_cosine', col='variant'),
           'C':   dict(voice=R4 + ['whisper'], p_id=64, cosine=True,  nap=True,  csv='nb18_cv.csv', variant='V1_batch64_cosine', col='variant'),
           # [v5 3-Nov] D = C + InfoNCE alignment (w 0.5, tau 0.07) with C's final recipe (64 ids/batch, cosine LR, C's epochs)  -> dev 22.70
           'D':   dict(voice=R4 + ['whisper'], p_id=64, cosine=True,  nap=True,  csv='nb18_cv.csv', variant='V1_batch64_cosine', col='variant', align='nce', w=0.5, tau=0.07),
           # [v5 3-Nov] D16 = the exact NB28 CV recipe (16 ids/batch, constant LR, epochs = median best epoch of NB28 H2)  -> dev 23.32
           'D16': dict(voice=R4 + ['whisper'], p_id=16, cosine=False, nap=True,  csv='nb28_cv.csv', variant='H2_align_nce', col='combo', align='nce', w=0.5, tau=0.07),
           # [v6 4-Nov] E = D with the SupCon L_out loss form (NB33 L1_lout)  -> dev 22.16 (current entry)
           'E':   dict(E_BASE),
           # [v6 4-Nov] E2 = E + VoxCeleb2 part 2 identities (NB32)  -> dev 21.92
           'E2':  dict(E_BASE, extra_vox=['vox2_mp4_2']),
           # [v7 5-Nov] G = E2 data + S28 head settings, with the final 64-id cosine recipe and E's epochs  -> dev 22.35
           'G':   dict(voice=R4 + ['whisper'], p_id=64, cosine=True,  nap=True,  csv='nb18_cv.csv', variant='V1_batch64_cosine', col='variant', extra_vox=['vox2_mp4_2'], **S28),
           # [v7 5-Nov] G16 = E2 data + S28 exactly as in CV (16 ids/batch, constant LR, epochs = median best epoch of S28 in NB34)  -> dev 23.07
           'G16': dict(voice=R4 + ['whisper'], p_id=16, cosine=False, nap=True,  csv='nb34_cv.csv', variant='S28', col='combo', extra_vox=['vox2_mp4_2'], **S28),
           # [v8 5-Nov] E3 = E + VoxCeleb2 parts 2 and 3 (NB32 + NB32b), about 3,700 training identities
           'E3':  dict(E_BASE, extra_vox=['vox2_mp4_2', 'vox2_mp4_3']),
           # [v8 5-Nov] E_nap24 = E with 24 language directions removed instead of 8
           'E_nap24': dict(E_BASE, nap_k=24),
           # [v10 5-Nov] research round 9: VoxCeleb carries nationality/accent directions that are useless in a single-population test set, and our
           # gender-Bangla cell worsened monotonically with VoxCeleb size. E_novox = E without VoxCeleb; E_sel500 / E_sel900 = E trained with only the
           # 500 / 900 of the ~3,700 VoxCeleb identities (4 parts) whose face + voice centroids are nearest to the MAV + challenge training population
           'E_novox':  dict(E_BASE, vox_parts=[]),
           'E_sel500': dict(E_BASE, vox_parts=['vox2_test_mp4', 'vox2_mp4_1', 'vox2_mp4_2', 'vox2_mp4_3'], vox_select=500),
           'E_sel900': dict(E_BASE, vox_parts=['vox2_test_mp4', 'vox2_mp4_1', 'vox2_mp4_2', 'vox2_mp4_3'], vox_select=900),
           # [v11 7-Nov] J = E's recipe and data with NB41's LoRA-tuned CLIP (features/clip_ft.npz, vox_<tag>_clip_ft.npy, encoder final/clip_ft.pt) as the face block.
           # NB41 CV vs E's recipe: stage A -0.50, stage B -0.14 (std -0.05, gender -0.22); under the -0.4 bar, so judged on the dev board against E (22.16)
           'J':   dict(E_BASE, face='clip_ft'),
           'B':   dict(voice=['ecapa', 'wavlm_sv', 'r293'], p_id=16, cosine=False, nap=False, csv='nb15_cv.csv', variant='P1_proto', col='variant')}
print('[v11 7-Nov] ready')


In [ ]:
# Cell 3 [v11 7-Nov] - train the FINAL heads once from cached training features and save them (skipped if present); eval mode loads the entry only
# v11: cfg 'face' picks the face block per system (training groups are keyed on it); 'clip_ft' rows come from features/clip_ft.npz (train/mav/extra) and
#      features/vox_<tag>_clip_ft.npy (NB41). Dev mode now scores E, E2 and J only (every other system is rejected and its zip exists).
# v10: cfg 'vox_parts' = explicit list of VoxCeleb parts (overrides VOX_BASE + extra_vox; [] = no VoxCeleb); cfg 'vox_select' = N keeps only the N VoxCeleb
#      identities whose face (CLIP) and voice (ECAPA) centroids are nearest to the MAV + challenge training population (research round 9: VoxCeleb
#      carries nationality/accent directions that are useless inside a single-population test set; training-side curation, declared)
# v3: 21 seeds, system A only. v4: + C (Whisper + NAP-8). v5: + D, D16 (InfoNCE). v6: + E (SupCon L_out), E2 (E + VoxCeleb2 part 2).
# v7: + G, G16 (E2 data + NB34 S28 head settings). v8: + E3 (VoxCeleb2 parts 2 + 3, NB32b), E_nap24 (NAP k 24); NB37 hooks (sf2 / hinge / pair / mix) by cfg.
# Systems with different VoxCeleb parts are trained in separate groups, each with its own training table. Bundles: head_<k>_s21.pt.
N_SEEDS = 21
VOX_BASE = ['vox2_test_mp4', 'vox2_mp4_1']
def vox_ready(tag):
    return os.path.exists(os.path.join(FEATDIR, 'vox_%s.npz' % tag)) and any(os.path.exists(os.path.join(FEATDIR, 'vox_%s_whisper.%s' % (tag, ext))) for ext in ('npy', 'npz'))
NB32_DONE, NB32B_DONE = vox_ready('vox2_mp4_2'), vox_ready('vox2_mp4_3')
FT_READY = os.path.exists(os.path.join(FEATDIR, 'clip_ft.npz')) and all(os.path.exists(os.path.join(FEATDIR, 'vox_%s_clip_ft.npy' % t)) for t in VOX_BASE)   # [v11] NB41 outputs
ACTIVE = ['E'] + (['E2'] if NB32_DONE else []) + (['J'] if FT_READY else [])
if MODE == 'eval':
    ACTIVE = [EVAL_ENTRY]                              # [v9] the evaluation entry only: one head bundle, one zip, nothing else to confuse
    assert os.path.exists(os.path.join(FINAL, 'head_%s_s%d.pt' % (EVAL_ENTRY, N_SEEDS))), 'head_%s_s21.pt is missing from FLAG2027/final/ - tell Claude' % EVAL_ENTRY
print('VoxCeleb part 2 features found:', NB32_DONE, '| part 3 found:', NB32B_DONE, '| LoRA-CLIP features found:', FT_READY, '| active systems:', ACTIVE)
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
def bundle_path(k):
    return os.path.join(FINAL, 'head_%s_s%d.pt' % (k, N_SEEDS))
need_all = [k for k in ACTIVE if not os.path.exists(bundle_path(k))]
groups = {}
def vox_key(cfg):
    return (tuple(cfg['vox_parts']) if 'vox_parts' in cfg else tuple(VOX_BASE + list(cfg.get('extra_vox', []))), int(cfg.get('vox_select', 0)), cfg.get('face', 'clip'))
for k in need_all:
    groups.setdefault(vox_key(SYSTEMS[k]), []).append(k)
for (parts_, n_sel, face_b), need in groups.items():
    VOX_TAGS = list(parts_)
    print('training group %s with face block %s and VoxCeleb parts %s%s' % (need, face_b, VOX_TAGS, ' | keep the %d nearest identities' % n_sel if n_sel else ''))
    BLOCKS = sorted({v for k in need for v in SYSTEMS[k]['voice']} - {'ecapa'})          # cached per-block feature files to load
    for b in BLOCKS:
        assert os.path.exists(os.path.join(FEATDIR, b + '.npz')), 'missing features/%s.npz' % b
    LOCAL = '/content/flag_small'; os.makedirs(LOCAL, exist_ok=True)
    if not os.path.exists(os.path.join(LOCAL, 'train.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(os.path.join(LOCAL, 'train.done'), 'w').close()
    tv = [p for p in glob.glob(LOCAL + '/**/train_English_voices.csv', recursive=True)][0]
    Xv, y = read_feat(tv, 192); y = y.astype(int)
    CHF = {'ecapa': Xv}
    for name in ['clip'] + ([face_b] if face_b != 'clip' else []) + BLOCKS:
        CHF[name] = np.load(os.path.join(FEATDIR, name + '.npz'))['train']
    MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
    for name in BLOCKS:
        if name in ('wavlm_sv',):
            continue                                   # already inside the mavceleb_* files
        d = np.load(os.path.join(FEATDIR, name + '.npz'))
        if name != 'r293' or 'mav' in d.files:
            MAV[name] = d['mav']
        if name != 'r293':
            EXTRA[name] = d['extra']
    if face_b != 'clip':
        d = np.load(os.path.join(FEATDIR, face_b + '.npz')); MAV[face_b] = d['mav']; EXTRA[face_b] = d['extra']          # [v11] fine-tuned face block rows (NB41)
    VOX = {k: [] for k in ['spk', 'clip', 'ecapa', 'wavlm_sv'] + ([face_b] if face_b != 'clip' else []) + [b for b in BLOCKS if b != 'wavlm_sv']}
    for tag in VOX_TAGS:
        d = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
        for b in BLOCKS:
            if b == 'wavlm_sv':
                continue
            pn, pz = os.path.join(FEATDIR, 'vox_%s_%s.npy' % (tag, b)), os.path.join(FEATDIR, 'vox_%s_%s.npz' % (tag, b))
            if os.path.exists(pn):
                d[b] = np.load(pn)
            elif os.path.exists(pz):
                d[b] = np.load(pz)['h']
            else:
                raise FileNotFoundError('VoxCeleb features for block %s / %s missing (run NB26 / NB32 / NB32b)' % (b, tag))
        if face_b != 'clip':
            pf = os.path.join(FEATDIR, 'vox_%s_%s.npy' % (tag, face_b)); assert os.path.exists(pf), 'VoxCeleb %s features for %s missing (NB41 Cell 4 writes them)' % (face_b, tag); d[face_b] = np.load(pf)
        for k in VOX:
            VOX[k].append(d[k])
        print('  VoxCeleb %-14s %6d clips, %5d identities' % (tag, len(d['spk']), len(np.unique(d['spk']))))
        del d
    if VOX_TAGS:
        VOX = {k: np.concatenate(v) for k, v in VOX.items()}
    else:
        VOX = {k: (np.array([], dtype=str) if k == 'spk' else np.zeros((0, (CHF if k in CHF else MAV)[k].shape[1]), np.float32)) for k in VOX}   # [v10] no VoxCeleb at all
    if n_sel:
        # [v10] identity selection: per-identity l2 centroids of face CLIP and voice ECAPA; score = mean of the nearest-cosine to any MAV / challenge
        # training identity in each modality; keep the n_sel VoxCeleb identities with the highest score (training data only, nothing from test)
        def cents(X, S):
            u = np.unique(S); return u, l2(np.stack([l2(X[S == s]).mean(0) for s in u]))
        tgt_spk = np.array(list(MAV['spk']) + list(EXTRA['spk']) + ['ch_%d' % c for c in y])
        _, TF = cents(np.concatenate([MAV['clip'], EXTRA['clip'], CHF['clip']]), tgt_spk); _, TV = cents(np.concatenate([MAV['ecapa'], EXTRA['ecapa'], CHF['ecapa']]), tgt_spk)
        vid, VF = cents(VOX['clip'], VOX['spk']); _, VV = cents(VOX['ecapa'], VOX['spk'])
        sim = 0.5 * ((VF @ TF.T).max(1) + (VV @ TV.T).max(1)); order = np.argsort(-sim); keep_ids = set(vid[order[:n_sel]])
        m = np.isin(VOX['spk'], list(keep_ids)); VOX = {k: v[m] for k, v in VOX.items()}
        note('  VoxCeleb selection: kept %d of %d identities (%d clips) nearest to the MAV + challenge population | similarity kept %.3f-%.3f, dropped %.3f-%.3f' % (
            len(keep_ids), len(vid), int(m.sum()), sim[order[n_sel - 1]], sim[order[0]], sim[order[-1]], sim[order[n_sel]] if n_sel < len(vid) else float('nan')))
    N_CH, N_MAV, N_EX, N_VOX = len(y), len(MAV['spk']), len(EXTRA['spk']), len(VOX['spk'])
    ext_spk = list(MAV['spk']) + list(EXTRA['spk']) + list(VOX['spk'])
    print('  speaker-name overlap VoxCeleb vs MAV (should be 0):', len((set(VOX['spk']) & set(MAV['spk'])) | (set(VOX['spk']) & set(EXTRA['spk']))))
    ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
    Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
    D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV + N_EX, int), 2 * np.ones(N_VOX, int)]
    F_RAW = np.concatenate([l2(CHF[face_b]), l2(MAV[face_b]), l2(EXTRA[face_b]), l2(VOX[face_b])]).astype(np.float32)   # [v11] face block per group
    MAV_LANG = None
    if any(SYSTEMS[k].get('nap') for k in need):
        ml = np.load(os.path.join(FEATDIR, 'mav_lang.npz'))
        assert list(ml['spk']) == ext_spk[:N_MAV + N_EX], 'mav_lang.npz row order differs from the MAV feature files - tell Claude'
        MAV_LANG = (ml['spk'], ml['lang'])
    def voice_table(names):
        return np.concatenate([np.concatenate([l2(src[n]) for n in names], 1) for src in (CHF, MAV, EXTRA, VOX)]).astype(np.float32)
    SEEDS, K_CLIP, ALPHA, LR, STEPS, IN_FRAC = list(range(1, N_SEEDS + 1)), 8, 2.0, 1e-4, 60, 0.75
    def prepare_voice(cfg):
        V_RAW = voice_table(cfg['voice']); U = None
        if cfg.get('nap'):
            U, n_d = nap_basis(V_RAW[N_CH:N_CH + N_MAV + N_EX], MAV_LANG[0], MAV_LANG[1], k=cfg.get('nap_k', NAP_K))
            print('  NAP basis: %d language-difference vectors -> top %d directions removed from the %d-d voice vector' % (n_d, U.shape[0], V_RAW.shape[1]))
            V_RAW = nap_apply(V_RAW, U)
        return V_RAW, U
    def train_final(cfg, V_RAW, seed, epochs):
        torch.manual_seed(seed); rng = np.random.default_rng(seed)
        F, V = F_RAW.copy(), V_RAW.copy(); ch = None
        for d in np.unique(D_ALL):
            rows = np.where(D_ALL == d)[0]
            st = (F_RAW[rows].mean(0), F_RAW[rows].std(0) + 1e-6, V_RAW[rows].mean(0), V_RAW[rows].std(0) + 1e-6)
            F[rows] = (F_RAW[rows] - st[0]) / st[1]; V[rows] = (V_RAW[rows] - st[2]) / st[3]
            if d == 0:
                ch = st
        XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev); del F, V
        cl = np.unique(Y_ALL); rm = {c: i for i, c in enumerate(cl)}
        ytr_np = np.array([rm[c] for c in Y_ALL]); ytr = torch.tensor(ytr_np, device=dev)
        cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL < 2])] = True
        in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(cfg['p_id'] * IN_FRAC))
        pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
        model = make_model(cfg, len(cl), XF.shape[1], XV.shape[1]).to(dev)
        model.norm = (XF.mean(0), XF.std(0) + 1e-6, XV.mean(0), XV.std(0) + 1e-6); mf, sf_, mv, sv = model.norm
        opt = torch.optim.Adam(model.parameters(), lr=cfg.get('lr', LR), weight_decay=0.01)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100 * STEPS) if cfg['cosine'] else None   # same schedule length as CV runs
        align, alpha, p_in = ALIGN_FN.get(cfg.get('align')), cfg.get('alpha', ALPHA), cfg.get('p_in', 0.0)
        use_sf2, w_h, use_pair, mix_p = cfg.get('sf2', False), cfg.get('hinge', 0.0), cfg.get('pair', False), cfg.get('mix', 0.0)   # [v8] NB37 hooks
        for ep in range(epochs):
            model.train()
            for _ in range(STEPS):
                if len(out_cls) == 0:
                    chosen = rng.choice(in_cls, cfg['p_id'], replace=False)                       # [v10] no VoxCeleb rows (E_novox)
                else:
                    chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, cfg['p_id'] - n_in, replace=len(out_cls) < cfg['p_id'] - n_in)]
                j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]); j = torch.tensor(j_np, device=dev)
                xf = (XF[j] - mf) / sf_; xv = (XV[j] - mv) / sv; lab = ytr[j]
                if p_in > 0:
                    xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)          # [v7] input dropout (training only)
                if mix_p > 0:
                    # [v8] partner = another row of the same identity (the K_CLIP rows of each identity are contiguous); shared lambda per row
                    part = torch.tensor(np.concatenate([g[rng.permutation(K_CLIP)] for g in np.arange(len(j_np)).reshape(cfg['p_id'], K_CLIP)]), device=dev)
                    lam = torch.tensor(rng.beta(2.0, 2.0, size=len(j_np)), device=dev, dtype=torch.float32)
                    lam = torch.where(torch.tensor(rng.random(len(j_np)) < mix_p, device=dev), lam, torch.ones_like(lam))[:, None]
                    xf = lam * xf + (1 - lam) * xf[part]; xv = lam * xv + (1 - lam) * xv[part]
                fused, logits, ft, vt = model(xf, xv)
                cls_term = sf2_loss(model.cls(fused), model.P, lab, model.sf2_bias) if use_sf2 else Fn.cross_entropy(logits, lab)
                loss = cls_term + alpha * opl_loss(fused, lab)
                if align is not None:
                    loss = loss + cfg['w'] * align(ft, vt, lab, cfg['tau'])          # D/D16: InfoNCE L_in | E/E2/E3/G: SupCon L_out
                if w_h > 0:
                    loss = loss + w_h * hinge_loss(ft, vt)
                if use_pair:
                    loss = loss + pair_bce(model.pair, ft, vt, lab)
                opt.zero_grad(); loss.backward(); opt.step()
                if sched is not None:
                    sched.step()
        return model, ch
    for k in need:
        cfg = SYSTEMS[k]; R = pd.read_csv(os.path.join(RESDIR, cfg['csv']))
        epochs = int(np.median(R[R[cfg['col']] == cfg['variant']]['best_ep']))
        V_RAW, U = prepare_voice(cfg)
        bundle = {'cfg': cfg, 'epochs': epochs, 'n_class': int(len(np.unique(Y_ALL))), 'd_face': int(F_RAW.shape[1]), 'd_voice': int(V_RAW.shape[1]), 'nap': U, 'vox_tags': VOX_TAGS, 'face': face_b, 'models': []}
        t0 = time.time()
        for s in SEEDS:
            m, ch = train_final(cfg, V_RAW, s, epochs)
            bundle['models'].append({'state': {kk: v.cpu() for kk, v in m.state_dict().items()}, 'norm': [t.cpu() for t in m.norm], 'ch': [np.asarray(c) for c in ch]})
            del m; torch.cuda.empty_cache()
            if s % 7 == 0:
                print('  head %s: %d / %d seeds, %.0f min' % (k, s, N_SEEDS, (time.time() - t0) / 60))
        torch.save(bundle, bundle_path(k))
        hooks = ', '.join(h for h in ['sf2', 'hinge', 'pair', 'mix'] if cfg.get(h))
        note('head %s trained (face %s | %s%s + %s%s; width %d, branch dropout %.1f; %d ids/batch, %d training identities from VoxCeleb parts %s, %d epochs x %d seeds, %.0f min) and saved as %s' % (k, face_b, '+'.join(cfg['voice']), ' + NAP%d' % U.shape[0] if U is not None else '', cfg.get('align', 'no align'), ' + ' + hooks if hooks else '', cfg.get('d', 128), cfg.get('p_br', 0.5), cfg['p_id'], len(np.unique(Y_ALL)), VOX_TAGS, epochs, N_SEEDS, (time.time() - t0) / 60, os.path.basename(bundle_path(k))))
        del V_RAW
HEADS = {}
for k in ACTIVE:
    b = torch.load(bundle_path(k), map_location=dev, weights_only=False); ms = []
    for mb in b['models']:
        m = make_model(b['cfg'], b['n_class'], b['d_face'], b['d_voice']).to(dev); m.load_state_dict(mb['state']); m.norm = [t.to(dev) for t in mb['norm']]; m.ch = mb['ch']; ms.append(m.eval())
    HEADS[k] = (b['cfg'], ms, b.get('nap'))
note('[v11 7-Nov] heads loaded: ' + ', '.join('%s (face %s, %s%s, width %d, %d seeds)' % (k, HEADS[k][0].get('face', 'clip'), HEADS[k][0].get('align', 'no align'), ', NAP%d' % HEADS[k][2].shape[0] if HEADS[k][2] is not None else '', HEADS[k][0].get('d', 128), len(HEADS[k][1])) for k in HEADS))


In [ ]:
# Cell 4 [v11 7-Nov] - encoders for RAW files (identical definitions to the extraction notebooks; + Whisper-v3 encoder from NB24; v11: + NB41's LoRA-tuned CLIP)
from huggingface_hub import hf_hub_download, list_repo_files
try:
    from speechbrain.inference.interfaces import Pretrained
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]
def enc_ecapa(paths, bs=16):
    E = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); E.eval()
    out = np.zeros((len(paths), 192), np.float32)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='ecapa'):
        wavs = [librosa.load(paths[j], sr=16000, mono=True)[0] for j in idx]
        lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max()); b = torch.zeros(len(wavs), L)
        for k, w in enumerate(wavs):
            b[k, :len(w)] = torch.from_numpy(w)
        with torch.no_grad():
            out[idx] = E.encode_batch(b, lens / L).squeeze(1).cpu().numpy()
    return out
def enc_clip(paths):
    from transformers import CLIPModel, CLIPProcessor
    name = 'openai/clip-vit-large-patch14'
    model = CLIPModel.from_pretrained(name).half().to(dev).eval(); proc = CLIPProcessor.from_pretrained(name); out = []
    for i in tqdm(range(0, len(paths), 64), desc='clip'):
        ims = [Image.open(p).convert('RGB') for p in paths[i:i + 64]]
        x = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
        with torch.no_grad():
            out.append(model.visual_projection(model.vision_model(pixel_values=x).pooler_output).float().cpu().numpy())
    return np.concatenate(out)
def enc_wavlm(paths, max_sec=10):
    from transformers import AutoFeatureExtractor, WavLMForXVector
    name = 'microsoft/wavlm-base-plus-sv'
    fe = AutoFeatureExtractor.from_pretrained(name); model = WavLMForXVector.from_pretrained(name).to(dev).eval(); out = []
    for i in tqdm(range(0, len(paths), 32), desc='wavlm'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + 32]]
        x = fe(wavs, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors='pt').to(dev)
        with torch.no_grad():
            out.append(model(**x).embeddings.cpu().numpy())
    return np.concatenate(out)
def enc_r293(paths, sec=6, bs=32):
    import torchaudio
    from onnx2torch import convert
    repo = 'Wespeaker/wespeaker-voxceleb-resnet293-LM'
    onnx_name = [f for f in list_repo_files(repo) if f.endswith('.onnx')][0]
    net = convert(hf_hub_download(repo, onnx_name, local_dir=os.path.join(CACHE, 'wespeaker_r293'))).to(dev).eval(); L = 16000 * sec
    def fbank(p):
        w = librosa.load(p, sr=16000, mono=True)[0]
        w = w[(len(w) - L) // 2:(len(w) - L) // 2 + L] if len(w) >= L else np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L]
        fb = torchaudio.compliance.kaldi.fbank(torch.from_numpy(w).unsqueeze(0) * (1 << 15), num_mel_bins=80, frame_length=25, frame_shift=10, dither=0.0, sample_frequency=16000, window_type='hamming', use_energy=False)
        return fb - fb.mean(0, keepdim=True)
    out = []
    for i in tqdm(range(0, len(paths), bs), desc='resnet293'):
        x = torch.stack([fbank(p) for p in paths[i:i + bs]]).to(dev)
        with torch.no_grad():
            e = net(x)
        e = e[0] if isinstance(e, (list, tuple)) else e; out.append(e.reshape(len(x), -1).float().cpu().numpy())
    return np.concatenate(out)
def enc_redim(paths, bs=16, max_sec=10):
    net = torch.hub.load('PalabraAI/redimnet2', 'redimnet2', model_name='b6', train_type='lm', dataset='vb2+vox2_v0', pretrained=True).to(dev).eval()
    order = np.argsort([sf.info(p).duration for p in paths]); emb = {}
    for i in tqdm(range(0, len(order), bs), desc='redimnet2'):
        idx = order[i:i + bs]; wavs = [librosa.load(paths[j], sr=16000, mono=True)[0][:16000 * max_sec] for j in idx]
        L = max(len(w) for w in wavs); b = torch.zeros(len(wavs), L)
        for k_, w in enumerate(wavs):
            b[k_] = torch.from_numpy(np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L])
        with torch.no_grad():
            e = net(b.to(dev))
        e = e[0] if isinstance(e, (list, tuple)) else e
        for k_, j in enumerate(idx):
            emb[int(j)] = e[k_].float().cpu().numpy().reshape(-1)
    return np.stack([emb[j] for j in range(len(paths))])
from transformers import Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2PreTrainedModel
class ModelHead(nn.Module):
    def __init__(self, hidden, num_labels, drop):
        super().__init__(); self.dense = nn.Linear(hidden, hidden); self.dropout = nn.Dropout(drop); self.out_proj = nn.Linear(hidden, num_labels)
    def forward(self, x):
        return self.out_proj(self.dropout(torch.tanh(self.dense(self.dropout(x)))))
class AgeGenderModel(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config); self.wav2vec2 = Wav2Vec2Model(config)
        self.age = ModelHead(config.hidden_size, 1, config.final_dropout); self.gender = ModelHead(config.hidden_size, 3, config.final_dropout); self.post_init()
    def forward(self, input_values):
        h = self.wav2vec2(input_values)[0].mean(1); return h, self.age(h), torch.softmax(self.gender(h), dim=1)
def enc_audeer(paths, bs=16, max_sec=8):
    name = 'audeering/wav2vec2-large-robust-24-ft-age-gender'; proc = Wav2Vec2Processor.from_pretrained(name)
    try:
        model = AgeGenderModel.from_pretrained(name)
    except Exception:
        base = Wav2Vec2Model.from_pretrained(name); model = AgeGenderModel(base.config); model.wav2vec2 = base
        try:
            from safetensors.torch import load_file; sd = load_file(hf_hub_download(name, 'model.safetensors'))
        except Exception:
            sd = torch.load(hf_hub_download(name, 'pytorch_model.bin'), map_location='cpu')
        model.load_state_dict({k: v for k, v in sd.items() if k.startswith(('age.', 'gender.'))}, strict=False)
    model = model.to(dev).eval().half(); H = []
    for i in tqdm(range(0, len(paths), bs), desc='audeering'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + bs]]
        x = proc(wavs, sampling_rate=16000, padding=True, return_tensors='pt')['input_values'].to(dev).half()
        with torch.no_grad():
            h, _, _ = model(x)
        H.append(h.float().cpu().numpy())
    return np.concatenate(H)
def enc_whisper(paths, bs=8, layers=(20, 29)):
    # Whisper-large-v3 ENCODER only (frozen, multilingual pretraining incl. Bengali -> allowed by the 28 Oct FAQ as a frozen pretrained audio model);
    # mean of hidden layers 20-28, pooled over the clip's real frames (20 ms each). Identical to NB24 / NB26.
    from transformers import WhisperFeatureExtractor, WhisperModel
    name = 'openai/whisper-large-v3'
    fe = WhisperFeatureExtractor.from_pretrained(name); enc = WhisperModel.from_pretrained(name, torch_dtype=torch.float16).encoder.to(dev).eval(); out = []
    for i in tqdm(range(0, len(paths), bs), desc='whisper-v3 enc'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * 30] for p in paths[i:i + bs]]
        x = fe(wavs, sampling_rate=16000, return_tensors='pt').input_features.to(dev, torch.float16)
        with torch.no_grad():
            hs = enc(x, output_hidden_states=True).hidden_states
        h = torch.stack(hs[layers[0]:layers[1]]).float().mean(0)
        for k_, w in enumerate(wavs):
            n = max(1, min(1500, len(w) // 320)); out.append(h[k_, :n].mean(0).cpu().numpy())
    del enc; torch.cuda.empty_cache(); return np.stack(out)
def enc_clip_ft(paths, bs=64):
    # [v11 7-Nov] NB41's LoRA-tuned CLIP ViT-L/14: the frozen model + rank-8 LoRA on q/v of every vision block, weights from final/clip_ft.pt (key 'lora',
    # the checkpoint NB41 selected on held-out external identities). Preprocessing = NB41's torchvision pipeline (bicubic resize 224, centre crop,
    # CLIP mean/std), verified against the stored CLIPProcessor rows at cosine 1.0000 in NB41. Dropout is inactive in eval mode.
    from transformers import CLIPModel
    from peft import LoraConfig, inject_adapter_in_model
    import torchvision.transforms as T
    from concurrent.futures import ThreadPoolExecutor
    model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(dev).eval()
    for p in model.parameters():
        p.requires_grad = False
    vis = inject_adapter_in_model(LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=['q_proj', 'v_proj']), model.vision_model)
    ck = torch.load(os.path.join(FINAL, 'clip_ft.pt'), map_location=dev, weights_only=False)
    res = vis.load_state_dict(ck['lora'], strict=False)
    n_lora = sum('lora_' in k for k in vis.state_dict())
    assert len(ck['lora']) == n_lora and not res.unexpected_keys and not any('lora_' in k for k in res.missing_keys), 'LoRA weights did not load cleanly (%d saved, %d in the model, %d unexpected, %d lora keys missing) - tell Claude' % (len(ck['lora']), n_lora, len(res.unexpected_keys), sum('lora_' in k for k in res.missing_keys))
    vis.eval(); proj = model.visual_projection
    print('clip_ft: %d LoRA tensors loaded from final/clip_ft.pt (NB41 step %s, held-out external EER %.2f)' % (n_lora, ck.get('step'), ck.get('val', {}).get('val_ext', float('nan'))))
    tf = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(), T.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])])
    pool = ThreadPoolExecutor(8); out = []
    def load_img(p):
        return tf(Image.open(p).convert('RGB'))
    for i in tqdm(range(0, len(paths), bs), desc='clip_ft'):
        x = torch.stack(list(pool.map(load_img, paths[i:i + bs]))).to(dev)
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
            e = proj(vis(pixel_values=x).pooler_output)
        out.append(e.float().cpu().numpy())
    del model, vis; torch.cuda.empty_cache(); return np.concatenate(out)
ENC = {'clip': ('faces', enc_clip), 'clip_ft': ('faces', enc_clip_ft), 'ecapa': ('voices', enc_ecapa), 'wavlm_sv': ('voices', enc_wavlm), 'r293': ('voices', enc_r293), 'redim': ('voices', enc_redim), 'audeer': ('voices', enc_audeer), 'whisper': ('voices', enc_whisper)}
print('[v11 7-Nov] encoders defined:', list(ENC))

In [ ]:
# Cell 5 [v12 11-Nov] - locate the pair lists (dev rehearsal, or the ONE zip in FLAG2027/eval/) and resolve the raw files
# v9: everything an eval run writes is tagged with the zip's name (ZTAG), so a dry run on a renamed copy of dev_set.zip can never be cached as,
#     or mistaken for, the real evaluation set; exactly one zip allowed in eval/; list detection falls back to every .txt; file columns picked by extension
# v12: the organizers' evaluation_phase/gender/English_valid.txt has a space instead of '/' inside the face path
#      ("English_valid/faces 00000.jpg"). fix_row rejoins path pieces that were split by whitespace; a repaired path is used only if that file exists.
ROOT = '/content/data_' + MODE; os.makedirs(ROOT, exist_ok=True)
if MODE == 'dev':
    zips = [os.path.join(ZIP_DIR, 'dev_set.zip')]; ZTAG = 'dev'
else:
    zips = sorted(glob.glob(os.path.join(PROJECT, 'eval', '*.zip')))
    assert len(zips) == 1, 'FLAG2027/eval/ must contain exactly ONE zip, found %s. Remove the dry-run copy before the real evaluation zip goes in.' % [os.path.basename(z) for z in zips]
    ZTAG = 'eval_' + re.sub(r'[^A-Za-z0-9_-]+', '_', os.path.splitext(os.path.basename(zips[0]))[0])
note('MODE %s | zip %s (%.0f MB) | output tag %s' % (MODE, os.path.basename(zips[0]), os.path.getsize(zips[0]) / 1e6, ZTAG))
for z in zips:
    flag = os.path.join(ROOT, os.path.basename(z) + '.done')
    if not os.path.exists(flag):
        shutil.copy(z, '/content/tmp.zip')
        with zipfile.ZipFile('/content/tmp.zip') as zf:
            zf.extractall(ROOT)
        os.remove('/content/tmp.zip'); open(flag, 'w').close()
def classify(p):
    s = p.lower(); track = 'no_gender' if 'no_gender' in s else ('gender' if 'gender' in s else None)
    lang = 'English' if 'english' in s else ('Bangla' if ('bangla' in s or 'bengali' in s) else None)
    return track, lang
alltxt = glob.glob(ROOT + '/**/*.txt', recursive=True)
lists = [p for p in alltxt if 'test' in os.path.basename(p).lower() or 'eval' in os.path.basename(p).lower()]
if len({classify(p) for p in lists if all(classify(p))}) != 4:
    lists = alltxt                                     # [v9] the evaluation lists may carry neither 'test' nor 'eval' in their names
print('candidate lists:'); [print('   ', p[len(ROOT):]) for p in lists]
LISTS = {}
for p in lists:
    t, g = classify(p)
    if t and g:
        assert t + '_' + g not in LISTS, 'two files look like the %s_%s list: %s and %s - paste this to Claude' % (t, g, LISTS[t + '_' + g][len(ROOT):], p[len(ROOT):])
        LISTS[t + '_' + g] = p
assert len(LISTS) == 4, 'Expected 4 lists (no_gender/gender x English/Bangla), found %s among %s - paste this to Claude' % (list(LISTS), [p[len(ROOT):] for p in alltxt][:20])
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(ROOT):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
    return None
AUD, IMG = ('.wav', '.flac', '.mp3', '.m4a'), ('.jpg', '.jpeg', '.png')
def fix_row(r, list_path):
    """[v12] id + file paths; pieces without a file extension are glued to the next piece with '/' (or ' ' if that is the real name)"""
    out, buf = [r[0]], []
    for x in r[1:]:
        buf.append(x)
        if x.lower().endswith(AUD + IMG):
            if len(buf) == 1:
                out.append(x)
            else:
                cands = ['/'.join(buf), ' '.join(buf)]
                out.append(next((c for c in cands if resolve(c, list_path)), cands[0]))
            buf = []
    return out + buf
def pick(r, exts, default):
    t = [x for x in r if x.lower().endswith(exts)]; return t[0] if len(t) == 1 else r[default]      # dev format: id voice face
PAIRS = {}
for split, lp in LISTS.items():
    raw = [l.split() for l in open(lp) if l.strip()]
    rows = [fix_row(r, lp) for r in raw]
    n_fixed = sum(len(a) != len(b) for a, b in zip(raw, rows))
    if n_fixed:
        note('%s: %d of %d lines had a path split by a space; repaired, e.g. %s' % (split, n_fixed, len(rows), ' '.join(rows[0])[:90]))
    assert rows and all(len(r) >= 3 for r in rows), '%s: lines do not have 3 columns (id voice face), e.g. %r - paste this to Claude' % (split, rows[0] if rows else None)
    ids = [r[0] for r in rows]; vp = [resolve(pick(r, AUD, 1), lp) for r in rows]; fp = [resolve(pick(r, IMG, 2), lp) for r in rows]
    miss = [i for i, (a, b) in enumerate(zip(vp, fp)) if a is None or b is None]
    assert not miss, '%s: %d of %d pairs have a missing file, first at line %d: %s - paste this to Claude' % (split, len(miss), len(rows), miss[0] + 1, ' '.join(rows[miss[0]]))
    assert len(set(ids)) == len(ids), '%s: duplicate pair ids - paste this to Claude' % split
    # official ECAPA CSV next to the list, if provided (same encoder: verified cosine 0.9999)
    csvs = [c for c in glob.glob(os.path.dirname(lp) + '/**/*voices*.csv', recursive=True) if split.split('_')[-1] in os.path.basename(c)]
    PAIRS[split] = (ids, vp, fp, csvs[0] if csvs else None)
    note('%-18s %5d pairs | official ECAPA csv: %s | list %s | first line: %s' % (split, len(ids), 'yes' if csvs else 'no -> will compute', lp[len(ROOT):], ' '.join(rows[0])[:90]))

In [ ]:
# Cell 6 [v11 7-Nov] - extract features (v11: the face block(s) the active systems use, 'clip' and/or 'clip_ft') from the RAW files (cached per split in FLAG2027/final/ under the run's tag; only the blocks the ACTIVE systems need;
# an existing cache is completed with any missing block without recomputing the others). Dev cache names are unchanged (feats_dev_<split>.npz).
NEEDED = sorted({HEADS[k][0].get('face', 'clip') for k in HEADS}) + sorted({v for k in HEADS for v in HEADS[k][0]['voice']})
print('blocks needed:', NEEDED)
FEAT = {}
for split, (ids, vp, fp, csv) in PAIRS.items():
    path = os.path.join(FINAL, 'feats_%s_%s.npz' % (ZTAG, split))
    d = dict(np.load(path)) if os.path.exists(path) else {}
    d = {k: v for k, v in d.items() if len(v) == len(ids)}          # [v9] never reuse a cached block whose row count does not match this list
    missing = [n for n in NEEDED if n not in d]
    if not missing:
        FEAT[split] = d; note('%s: features loaded from cache %s' % (split, os.path.basename(path))); continue
    t0 = time.time()
    for name in missing:
        kind, fn = ENC[name]
        if name == 'ecapa' and csv is not None:
            d[name] = pd.read_csv(csv, header=None).iloc[:, :192].values.astype(np.float32)
        else:
            d[name] = fn(fp if kind == 'faces' else vp).astype(np.float32)
        assert len(d[name]) == len(ids), '%s/%s: %d rows for %d pairs - tell Claude' % (split, name, len(d[name]), len(ids))
        assert np.isfinite(d[name]).all(), '%s/%s: non-finite features - tell Claude' % (split, name)
    np.savez(path, **d); FEAT[split] = d
    note('%s: %s extracted from raw files in %.0f min -> %s' % (split, '+'.join(missing), (time.time() - t0) / 60, os.path.basename(path)))


In [ ]:
# Cell 7 [v11 7-Nov] - score with the frozen heads (v11: each system's own face block) (NAP applied for C/D/E*), validate, write ZIPs; in dev mode compare with earlier runs
# v6b: + 'E+E2' = raw average of the E and E2 scores (two of our own finished systems; no normalisation, no test statistics)
# v8: scorer chosen per system (pair-classifier logit for 'pair' systems, -L2 otherwise); each system's own NAP basis (k 8 or 24) is applied
# v9: outputs tagged with ZTAG; eval mode = system E only, dry-run check against the dev run of E when the pair counts match, line counts inside the zip
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
def system_scores(k):
    cfg, models, U = HEADS[k]; out = {}; scorer = score_fn(cfg)
    for split, d in FEAT.items():
        fx = l2(d[cfg.get('face', 'clip')]).astype(np.float32); vx = np.concatenate([l2(d[n]) for n in cfg['voice']], 1).astype(np.float32)
        if U is not None:
            vx = nap_apply(vx, U)                      # same fixed projection as in training (fitted on MAV training speakers only)
        sc = []
        for m in models:
            ch = m.ch; xf = torch.tensor((fx - ch[0]) / ch[1], device=dev, dtype=torch.float32); xv = torch.tensor((vx - ch[2]) / ch[3], device=dev, dtype=torch.float32)
            sc.append(scorer(m, xf, xv))
        out[split] = np.mean(sc, axis=0)
    return out
S = {k: system_scores(k) for k in HEADS}
if 'A' in S and 'C' in S:
    S['A+C'] = {s: (S['A'][s] + S['C'][s]) / 2 for s in FEAT}
    note('corr A vs C: ' + ', '.join('%s %.3f' % (s, np.corrcoef(S['A'][s], S['C'][s])[0, 1]) for s in FEAT))
if 'E' in S and 'E2' in S:
    S['E+E2'] = {s: (S['E'][s] + S['E2'][s]) / 2 for s in FEAT}
    note('corr E vs E2: ' + ', '.join('%s %.3f' % (s, np.corrcoef(S['E'][s], S['E2'][s])[0, 1]) for s in FEAT))
for other in ['E3', 'E_nap24', 'J']:
    if 'E' in S and other in S:
        note('corr E vs %s: ' % other + ', '.join('%s %.3f' % (s, np.corrcoef(S['E'][s], S[other][s])[0, 1]) for s in FEAT))
if MODE == 'dev' and 'A' in S:
    # same features, independently retrained heads: any gap from 1.0 is head variance (NB22: single-head sd ~0.5 EER). Read BEFORE this run saves its own scores.
    for ref_name in ['devscores_20_dev_s7_A.npz', 'devscores_18_V1_batch64_cosine.npz']:
        ref = os.path.join(RESDIR, ref_name)
        if os.path.exists(ref):
            R = dict(np.load(ref))
            note('corr A with %s: ' % ref_name + ', '.join('%s %.3f' % (s, np.corrcoef(S['A'][s], R[s])[0, 1]) for s in FEAT))
if MODE == 'eval':
    # [v9] dry-run check: if the zip has the dev set's pair counts, the scores must reproduce the dev run of the entry (same heads, same raw files)
    ref = os.path.join(RESDIR, 'devscores_20_dev_s%d_%s.npz' % (N_SEEDS, EVAL_ENTRY))
    if os.path.exists(ref):
        R = dict(np.load(ref))
        if all(s in R and len(R[s]) == len(S[EVAL_ENTRY][s]) for s in FEAT):
            note('DRY-RUN CHECK (pair counts equal the dev set): corr with the dev run of %s = ' % EVAL_ENTRY + ', '.join('%s %.4f' % (s, np.corrcoef(S[EVAL_ENTRY][s], R[s])[0, 1]) for s in FEAT) + '  | expected 1.0000 everywhere')
        else:
            note('pair counts differ from the dev set (%s vs dev %s) -> this is the real evaluation set; no dry-run check' % ({s: len(S[EVAL_ENTRY][s]) for s in FEAT}, {s: len(R[s]) for s in R}))
    else:
        note('no dev scores of %s found in results/ -> no dry-run check possible' % EVAL_ENTRY)
    for s in FEAT:
        v = S[EVAL_ENTRY][s]; note('scores %-18s n %5d | mean %.3f | sd %.3f | min %.3f | max %.3f | finite %s' % (s, len(v), v.mean(), v.std(), v.min(), v.max(), bool(np.isfinite(v).all())))
def write_zip(name, scores):
    OUT = '/content/sub_' + name; problems = []
    for split, (ids, vp, fp, csv) in PAIRS.items():
        track, lang_ = split.rsplit('_', 1); os.makedirs(os.path.join(OUT, track), exist_ok=True)
        sc = scores[split]
        if len(sc) != len(ids) or not np.isfinite(sc).all():
            problems.append(split)
        with open(os.path.join(OUT, track, LANG_FILE[lang_]), 'w') as f:
            for pid, v in zip(ids, -sc):                       # distance-like: LOWER = same person (confirmed on CodaBench)
                f.write('%s %.6f\n' % (pid, v))
    zp = os.path.join(SUBDIR, name + '.zip')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for track in ['no_gender', 'gender']:
            for fname in LANG_FILE.values():
                zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
    with zipfile.ZipFile(zp) as zf:
        names = sorted(zf.namelist()); counts = {n: sum(1 for _ in zf.open(n)) for n in names}
    ok = (not problems) and names == sorted(t + '/' + f for t in ['no_gender', 'gender'] for f in LANG_FILE.values())
    note('ZIP %-34s %s  (%s, %.1f MB) | lines: %s' % (name + '.zip', 'VALID' if ok else 'PROBLEM ' + str(problems), zp[len(BASE):], os.path.getsize(zp) / 1e6, ', '.join('%s %d' % (n, c) for n, c in counts.items())))
tag = '%s_s%d' % (ZTAG, N_SEEDS)
prefix = 'devscores' if MODE == 'dev' else 'scores'
for k in S:
    write_zip('20_%s_%s' % (tag, k.replace('+', '_')), S[k])
    np.savez(os.path.join(RESDIR, '%s_20_%s_%s.npz' % (prefix, tag, k.replace('+', '_'))), **S[k])
if MODE == 'eval':
    note('UPLOAD THIS ONE FILE: Drive > FLAG2027/submissions/20_%s_%s.zip' % (tag, EVAL_ENTRY))
else:
    for other in ['E_novox', 'E_sel500', 'E_sel900']:
        if 'E' in S and other in S:
            note('corr E vs %s: ' % other + ', '.join('%s %.3f' % (s, np.corrcoef(S['E'][s], S[other][s])[0, 1]) for s in FEAT))
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)


In [ ]:
# Cell 8 - DIAGNOSTIC (optional): is a drop in correlation due to feature drift (raw-file vs cached) or to head retraining?
if MODE == 'dev':
    print('%-10s %-18s %s' % ('encoder', 'split', 'median cosine(raw-file feature, cached feature)'))
    for name in ['clip', 'ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
        if name == 'ecapa' or not os.path.exists(os.path.join(FEATDIR, name + '.npz')):
            continue                                  # official CSV used in both -> identical by construction
        old = np.load(os.path.join(FEATDIR, name + '.npz'))
        for split in FEAT:
            if name not in FEAT[split] or split not in old.files:
                continue
            new = FEAT[split][name]; o = old[split]
            cos = (l2(new) * l2(o)).sum(1)
            flag = '' if np.median(cos) > 0.995 else '   <-- drift'
            print('%-10s %-18s %.4f (min %.3f)%s' % (name, split, np.median(cos), cos.min(), flag))
    # head-only effect: score the CACHED dev features with the frozen heads and compare with the cached-feature run
    cfg, models, _ = HEADS[next(iter(HEADS))]
    for split in FEAT:
        d = {n: np.load(os.path.join(FEATDIR, n + '.npz'))[split] for n in [cfg.get('face', 'clip')] + [v for v in cfg['voice'] if v != 'ecapa']}
        d['ecapa'] = FEAT[split]['ecapa']
        fx = l2(d[cfg.get('face', 'clip')]).astype(np.float32); vx = np.concatenate([l2(d[n]) for n in cfg['voice']], 1).astype(np.float32)
        sc = np.mean([head_score(m, torch.tensor((fx - m.ch[0]) / m.ch[1], device=dev, dtype=torch.float32), torch.tensor((vx - m.ch[2]) / m.ch[3], device=dev, dtype=torch.float32)) for m in models], axis=0)
        R = dict(np.load(os.path.join(RESDIR, 'devscores_18_V1_batch64_cosine.npz')))
        print('head-only  %-18s corr(frozen heads on CACHED features, NB18 run) = %.4f' % (split, np.corrcoef(sc, R[split])[0, 1]))

In [ ]:
# Cell 9 [v12] - eval-phase averaged systems. Run it AFTER eval runs with EVAL_ENTRY = E2, E and J. avg(E, E2) is the ranked FLAG 2027 submission (23.43).
# It averages scores that NB20 already saved (results/scores_20_eval_<zip>_s21_<system>.npz) with equal weights fixed before evaluation (FAQ 5),
# takes the pair ids in order from the E2 eval zip, and writes one new zip per average. No model or test statistic is touched.
import os, zipfile, numpy as np
P = '/content/drive/MyDrive/FLAG2027'; RES = os.path.join(P, 'results'); SUB = os.path.join(P, 'submissions')
TAG = 'eval_evaluation_phase_s21'          # change only if your eval zip is not called evaluation_phase.zip
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
FILES = [t + '/' + f for t in ['no_gender', 'gender'] for f in LANG_FILE.values()]
def split_of(name):
    t, f = name.split('/'); return '%s_%s' % (t, 'English' if 'English' in f else 'Bangla')
ref_zip = os.path.join(SUB, '20_%s_E2.zip' % TAG)
assert os.path.exists(ref_zip), 'run NB20 with EVAL_ENTRY = E2 first: %s not found' % ref_zip
IDS = {}
with zipfile.ZipFile(ref_zip) as zf:
    for n in FILES:
        IDS[split_of(n)] = [l.split()[0] for l in zf.read(n).decode().splitlines() if l.strip()]
def load(k):
    p = os.path.join(RES, 'scores_20_%s_%s.npz' % (TAG, k))
    return dict(np.load(p)) if os.path.exists(p) else None
for a, b in [('E', 'E2'), ('E2', 'J')]:
    A, B = load(a), load(b)
    if A is None or B is None:
        print('skip %s+%s: run NB20 with EVAL_ENTRY = %s first' % (a, b, a if A is None else b)); continue
    name = '20_%s_%s_%s' % (TAG, a, b); zp = os.path.join(SUB, name + '.zip'); ok = True
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for n in FILES:
            s = split_of(n); avg = (A[s] + B[s]) / 2
            ok &= len(avg) == len(IDS[s]) and bool(np.isfinite(avg).all())
            zf.writestr(n, ''.join('%s %.6f\n' % (pid, v) for pid, v in zip(IDS[s], -avg)))   # lower = same person
    print('ZIP %s.zip %s | corr %s vs %s: %s' % (name, 'VALID' if ok else 'PROBLEM - tell Claude', a, b,
          ', '.join('%s %.3f' % (s, np.corrcoef(A[s], B[s])[0, 1]) for s in IDS)))